In [ ]:
from datetime import datetime
from dotenv import load_dotenv
from dateutil.relativedelta import relativedelta  #is calender aware, as months are fixed No. days
from newsapi import NewsApiClient
import pandas as pd
import csv
import os


load_dotenv()
token = os.getenv('newsapi_TOKEN')
newsapi = NewsApiClient(api_key=token)

In [ ]:
def get_date_pairs():
    current_date = datetime.today().date()
    start_date = current_date - relativedelta(months=1)

    date_pairs = []

    while start_date<current_date:
        next_day_date = start_date + relativedelta(days=1)
        date_pairs.append((start_date,next_day_date))
        start_date = next_day_date

    return date_pairs

In [ ]:
def get_news_per_day():
    news_data = []
    for i in get_date_pairs():
        all_top_headlines = newsapi.get_everything(q='(oil OR crude OR energy OR brent OR wti) AND (iran OR price OR prices OR supply OR demand OR production OR disruption OR volatility OR surge OR drop)',
                                           from_param=i[0],
                                           to=i[1],
                                           language='en',
                                           sort_by='relevancy')
        articles = all_top_headlines['articles']
        news_data.extend(articles)
    return news_data
    
news_data = get_news_per_day()

In [ ]:
normalised_dataframe = pd.DataFrame(pd.json_normalize(news_data))
normalised_dataframe.drop_duplicates(subset=['title'],inplace=True)

def relevancy(row):
    text_to_run = (str(row['title']) + ' ' + str(row['description'])).lower()
    needed = ['oil','brent','wti','fuel','energy','opec','iran','strikes']
    return any(i in text_to_run for i in needed)

normalised_dataframe['relevant'] = normalised_dataframe.apply(relevancy, axis=1)
normalised_dataframe = normalised_dataframe[normalised_dataframe['relevant']==True]


In [5]:
if os.path.exists('NewsAPI_data_multipleday.csv'):
    normalised_dataframe.to_csv('NewsAPI_data_multipleday.csv', index=False, mode='a', header=False, encoding='utf-8')
    print('Dataset updated')

else:
    normalised_dataframe.to_csv('NewsAPI_data_multipleday.csv', index=False, encoding='utf-8')
    print('File does not exist, creating new')

Dataset updated
